In [5]:
from IPython.display import display, Math

%reload_ext autoreload
%autoreload 2

from qm_pomocne import (
    find_prime_implicants, 
    term_to_expression, 
    build_prime_implicant_chart, 
    find_essential_prime_implicants, 
    petricks_method
)

In [6]:
def quine_mccluskey(minterms: list[int], dont_cares: list[int] = None, variables: list[str] = None):
    """
    Minimizira boolean funkciju Quine-McCluskey algoritmom.

    Parameters:
        minterms: lista minterma funkcije
        dont_cares: lista don't care termina (zadano: prazna lista)
        variables: lista imena varijabli (zadano: A, B, C, ...)

    Returns:
        Lista minimalnih SOP izraza kao stringova, ili ['0'] ako je funkcija konstantno 0.
    """

    def validate_inputs():
        """Provjerava ispravnost ulaznih podataka i baca ValueError ako su neispravni."""
        for m in minterms:
            if m < 0:
                raise ValueError("Greška: Mintermi ne smiju biti negativni brojevi!")
        for d in dont_cares:
            if d < 0:
                raise ValueError("Greška: Don't care uvjeti ne smiju biti negativni brojevi!")
        for m in minterms:
            if m in dont_cares:
                raise ValueError(f"Greška: Broj {m} ne može biti istovremeno minterm i don't care!")

    def calculate_num_vars():
        """Izračunava minimalni broj varijabli potreban za prikaz svih minterma."""
        max_idx = 0
        for m in minterms:
            if m > max_idx: max_idx = m
        for d in dont_cares:
            if d > max_idx: max_idx = d

        num_vars = 1
        temp_idx = max_idx
        while temp_idx > 1:
            temp_idx = temp_idx // 2
            num_vars += 1
        return num_vars

    def resolve_variables(num_vars):
        """Vraća listu varijabli ili baca ValueError ako upisane varijable ne odgovaraju broju minterma."""
        if variables is None:
            result = []
            for i in range(num_vars):
                result.append(chr(65 + i))
            return result
        if len(variables) < num_vars:
            raise ValueError(f"Greška: Nedovoljno varijabli! Potrebne su barem {num_vars}, a upisano je {len(variables)}.")
        if len(variables) > num_vars:
            raise ValueError(f"Greška: Previše varijabli! Potrebne su točno {num_vars}, a upisano je {len(variables)}.")
        return variables

    def print_header(vars_resolved, num_vars):
        """Ispisuje zaglavlje s ulaznim podacima."""
        var_str = ""
        for i in range(len(vars_resolved)):
            var_str += vars_resolved[i]
            if i < len(vars_resolved) - 1: var_str += ", "

        print("=" * 55)
        print("         QUINE-McCLUSKEY ALGORITAM")
        print("=" * 55)
        print(f"Varijable   : {var_str}")
        print(f"Mintermi    : {minterms}")
        print(f"Don't cares : {dont_cares}")
        print(f"Broj varijabli: {num_vars}")

    def print_prime_implicants(prime_implicants, vars_resolved):
        """Ispisuje sve pronađene primarne implikante."""
        print("\n=== PRIMARNI IMPLIKANTI ===")
        for i in range(len(prime_implicants)):
            term = prime_implicants[i][0]
            mints = prime_implicants[i][1]
            expr = term_to_expression(term, vars_resolved)
            real_mints = [m for m in mints if m in minterms]
            print(f"  PI{i+1}: {term}  ({expr})  ←  mintermi {real_mints}")

    def print_essential_pis(essential_indices, covered, prime_implicants, vars_resolved):
        """Ispisuje esencijalne primarne implikante i minterme koje pokrivaju."""
        print("=== ESENCIJALNI PRIMARNI IMPLIKANTI ===")
        for idx in essential_indices:
            term = prime_implicants[idx][0]
            expr = term_to_expression(term, vars_resolved)
            print(f"  PI{idx+1}: {term}  ({expr})")
        covered_print = [m for m in minterms if m in covered]
        print(f"Pokriveni mintermi: {covered_print}")

    def build_final_expressions(additional_covers, essential_indices, prime_implicants, vars_resolved):
        """Spaja esencijalne i Petrickove PI-je u finalne SOP izraze i ispisuje ih."""
        sufiks = "I" if len(additional_covers) > 1 else ""
        print(f"\n=== MINIMALNI SOP IZRAZ{sufiks} ===")

        results = []
        for extra in additional_covers:
            all_pi_indices = []
            for idx in essential_indices:
                if idx not in all_pi_indices:
                    all_pi_indices.append(idx)
            for idx in extra:
                if idx not in all_pi_indices:
                    all_pi_indices.append(idx)
            all_pi_indices.sort()

            terms_str = ""
            for i in range(len(all_pi_indices)):
                idx = all_pi_indices[i]
                terms_str += term_to_expression(prime_implicants[idx][0], vars_resolved)
                if i < len(all_pi_indices) - 1:
                    terms_str += " + "

            latex_str = terms_str
            for v in vars_resolved:
                latex_str = latex_str.replace(v + "'", f"\\overline{{{v}}}")
            display(Math(f"f = {latex_str}"))
            results.append(terms_str)

        return results

    # ── Glavna logika ────────────────────────────────────────────────────────

    if dont_cares is None:
        dont_cares = []

    validate_inputs()
    num_vars = calculate_num_vars()
    vars_resolved = resolve_variables(num_vars)
    print_header(vars_resolved, num_vars)

    if len(minterms) == 0:
        print("\nFunkcija je konstantno 0.")
        return ['0']

    prime_implicants = find_prime_implicants(minterms, dont_cares, num_vars)
    print_prime_implicants(prime_implicants, vars_resolved)
    build_prime_implicant_chart(minterms, prime_implicants, vars_resolved)

    essential_indices, covered = find_essential_prime_implicants(minterms, prime_implicants)
    print_essential_pis(essential_indices, covered, prime_implicants, vars_resolved)

    additional_covers = petricks_method(minterms, prime_implicants, covered, essential_indices)
    results = build_final_expressions(additional_covers, essential_indices, prime_implicants, vars_resolved)

    print("\n" + "=" * 55)
    return results

In [ ]:
quine_mccluskey(
    minterms   = [0, 2, 3, 5, 6, 9, 10, 12, 15, 17, 18, 20, 23, 24, 27, 29, 30, 33, 34, 36, 39, 40, 43, 45, 46, 48, 51, 53, 54, 57, 58, 60, 63, 65, 66, 68, 71, 72, 75, 77, 78, 80, 83, 85, 86, 89, 90, 92, 95, 96, 99, 101, 102, 105, 106, 108, 111, 113, 114, 116, 119, 120, 123, 125, 126, 129, 130, 132, 135, 136, 139, 141, 142, 144, 147, 149, 150, 153, 154, 156, 159, 160, 163, 165, 166, 169, 170, 172, 175, 177, 178, 180, 183, 184, 187, 189, 190, 192, 195, 197, 198, 201, 202, 204, 207, 209, 210, 212, 215, 216, 219, 221, 222, 225, 226, 228, 231, 232, 235, 237, 238, 240, 243, 245, 246, 249, 250, 252, 255],
    dont_cares = [],
    variables  = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
)

In [14]:
quine_mccluskey(
    minterms   = [11, 38, 26, 10, 17, 40, 36, 60],
    dont_cares = [34, 20, 2]
    )

         QUINE-McCLUSKEY ALGORITAM
Varijable   : A, B, C, D, E, F
Mintermi    : [11, 38, 26, 10, 17, 40, 36, 60]
Don't cares : [34, 20, 2]
Broj varijabli: 6

--- Iteracija 1 ---
Termini (11):
  000010  ←  mintermi []  (don't cares [2])
  001010  ←  mintermi [10]
  001011  ←  mintermi [11]
  010001  ←  mintermi [17]
  010100  ←  mintermi []  (don't cares [20])
  011010  ←  mintermi [26]
  100010  ←  mintermi []  (don't cares [34])
  100100  ←  mintermi [36]
  100110  ←  mintermi [38]
  101000  ←  mintermi [40]
  111100  ←  mintermi [60]

--- Iteracija 2 ---
Termini (6):
  00-010  ←  mintermi [10]  (don't cares [2])
  -00010  ←  mintermi []  (don't cares [2, 34])
  00101-  ←  mintermi [10, 11]
  0-1010  ←  mintermi [10, 26]
  100-10  ←  mintermi [38]  (don't cares [34])
  1001-0  ←  mintermi [36, 38]

=== PRIMARNI IMPLIKANTI ===
  PI1: 010001  (A'BC'D'E'F)  ←  mintermi [17]
  PI2: 101000  (AB'CD'E'F')  ←  mintermi [40]
  PI3: 111100  (ABCDE'F')  ←  mintermi [60]
  PI4: 00-010  (A'B'D'EF'

<IPython.core.display.Math object>

["A'BC'D'E'F + AB'CD'E'F' + ABCDE'F' + A'B'CD'E + A'CD'EF' + AB'C'DF'"]

In [16]:
quine_mccluskey(
    minterms   = [1, 2, 3, 5, 9, 12, 14, 15],
    dont_cares = [4, 8, 11]
    )

         QUINE-McCLUSKEY ALGORITAM
Varijable   : A, B, C, D
Mintermi    : [1, 2, 3, 5, 9, 12, 14, 15]
Don't cares : [4, 8, 11]
Broj varijabli: 4

--- Iteracija 1 ---
Termini (11):
  0001  ←  mintermi [1]
  0010  ←  mintermi [2]
  0011  ←  mintermi [3]
  0100  ←  mintermi []  (don't cares [4])
  0101  ←  mintermi [5]
  1000  ←  mintermi []  (don't cares [8])
  1001  ←  mintermi [9]
  1011  ←  mintermi []  (don't cares [11])
  1100  ←  mintermi [12]
  1110  ←  mintermi [14]
  1111  ←  mintermi [15]

--- Iteracija 2 ---
Termini (13):
  00-1  ←  mintermi [1, 3]
  0-01  ←  mintermi [1, 5]
  -001  ←  mintermi [1, 9]
  001-  ←  mintermi [2, 3]
  -011  ←  mintermi [3]  (don't cares [11])
  010-  ←  mintermi [5]  (don't cares [4])
  -100  ←  mintermi [12]  (don't cares [4])
  100-  ←  mintermi [9]  (don't cares [8])
  1-00  ←  mintermi [12]  (don't cares [8])
  10-1  ←  mintermi [9]  (don't cares [11])
  1-11  ←  mintermi [15]  (don't cares [11])
  11-0  ←  mintermi [12, 14]
  111-  ←  mintermi

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

["A'C'D + A'B'C + BC'D' + ABC + B'D",
 "A'C'D + A'B'C + AC'D' + ABC + B'D",
 "A'C'D + A'B'C + ABD' + ABC + B'D",
 "A'C'D + A'B'C + ABD' + ACD + B'D",
 "A'B'C + A'BC' + BC'D' + ABC + B'D",
 "A'B'C + A'BC' + AC'D' + ABC + B'D",
 "A'B'C + A'BC' + ABD' + ABC + B'D",
 "A'B'C + A'BC' + ABD' + ACD + B'D"]

In [21]:
quine_mccluskey(
    minterms   = [1, 3, 6, 7, 11, 13, 14],
    dont_cares = [5, 12]
    )

         QUINE-McCLUSKEY ALGORITAM
Varijable   : A, B, C, D
Mintermi    : [1, 3, 6, 7, 11, 13, 14]
Don't cares : [5, 12]
Broj varijabli: 4

--- Iteracija 1 ---
Termini (9):
  0001  ←  mintermi [1]
  0011  ←  mintermi [3]
  0101  ←  mintermi []  (don't cares [5])
  0110  ←  mintermi [6]
  0111  ←  mintermi [7]
  1011  ←  mintermi [11]
  1100  ←  mintermi []  (don't cares [12])
  1101  ←  mintermi [13]
  1110  ←  mintermi [14]

--- Iteracija 2 ---
Termini (10):
  00-1  ←  mintermi [1, 3]
  0-01  ←  mintermi [1]  (don't cares [5])
  0-11  ←  mintermi [3, 7]
  -011  ←  mintermi [3, 11]
  01-1  ←  mintermi [7]  (don't cares [5])
  -101  ←  mintermi [13]  (don't cares [5])
  011-  ←  mintermi [6, 7]
  -110  ←  mintermi [6, 14]
  110-  ←  mintermi [13]  (don't cares [12])
  11-0  ←  mintermi [14]  (don't cares [12])

--- Iteracija 3 ---
Termini (1):
  0--1  ←  mintermi [1, 3, 7]  (don't cares [5])

=== PRIMARNI IMPLIKANTI ===
  PI1: -011  (B'CD)  ←  mintermi [3, 11]
  PI2: 011-  (A'BC)  ←  mi

<IPython.core.display.Math object>

<IPython.core.display.Math object>

["B'CD + BCD' + BC'D + A'D", "B'CD + BCD' + ABC' + A'D"]

In [22]:
quine_mccluskey(
    minterms   = [0, 1, 2, 5, 6, 7],
    dont_cares = []
    )

         QUINE-McCLUSKEY ALGORITAM
Varijable   : A, B, C
Mintermi    : [0, 1, 2, 5, 6, 7]
Don't cares : []
Broj varijabli: 3

--- Iteracija 1 ---
Termini (6):
  000  ←  mintermi [0]
  001  ←  mintermi [1]
  010  ←  mintermi [2]
  101  ←  mintermi [5]
  110  ←  mintermi [6]
  111  ←  mintermi [7]

--- Iteracija 2 ---
Termini (6):
  00-  ←  mintermi [0, 1]
  0-0  ←  mintermi [0, 2]
  -01  ←  mintermi [1, 5]
  -10  ←  mintermi [2, 6]
  1-1  ←  mintermi [5, 7]
  11-  ←  mintermi [6, 7]

=== PRIMARNI IMPLIKANTI ===
  PI1: 00-  (A'B')  ←  mintermi [0, 1]
  PI2: 0-0  (A'C')  ←  mintermi [0, 2]
  PI3: -01  (B'C)  ←  mintermi [1, 5]
  PI4: -10  (BC')  ←  mintermi [2, 6]
  PI5: 1-1  (AC)  ←  mintermi [5, 7]
  PI6: 11-  (AB)  ←  mintermi [6, 7]

=== TABLICA PRIMARNIH IMPLIKANTA ===
      | 0 | 1 | 2 | 5 | 6 | 7 | Pokriveni mintermi 
------+---+---+---+---+---+---+--------------------
A'B'  | X | X |   |   |   |   |        0, 1        
A'C'  | X |   | X |   |   |   |        0, 2        
B'C   |   

<IPython.core.display.Math object>

<IPython.core.display.Math object>

["A'B' + BC' + AC", "A'C' + B'C + AB"]

In [24]:
quine_mccluskey(
minterms   = [12, 13, 14, 28, 29, 45, 46, 54,
113, 114, 230, 231, 311, 405, 406, 450,
451, 501, 502, 510],
dont_cares = [15, 115, 310, 407],
variables  = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']
)

         QUINE-McCLUSKEY ALGORITAM
Varijable   : A, B, C, D, E, F, G, H, I
Mintermi    : [12, 13, 14, 28, 29, 45, 46, 54, 113, 114, 230, 231, 311, 405, 406, 450, 451, 501, 502, 510]
Don't cares : [15, 115, 310, 407]
Broj varijabli: 9

--- Iteracija 1 ---
Termini (24):
  000001100  ←  mintermi [12]
  000001101  ←  mintermi [13]
  000001110  ←  mintermi [14]
  000001111  ←  mintermi []  (don't cares [15])
  000011100  ←  mintermi [28]
  000011101  ←  mintermi [29]
  000101101  ←  mintermi [45]
  000101110  ←  mintermi [46]
  000110110  ←  mintermi [54]
  001110001  ←  mintermi [113]
  001110010  ←  mintermi [114]
  001110011  ←  mintermi []  (don't cares [115])
  011100110  ←  mintermi [230]
  011100111  ←  mintermi [231]
  100110110  ←  mintermi []  (don't cares [310])
  100110111  ←  mintermi [311]
  110010101  ←  mintermi [405]
  110010110  ←  mintermi [406]
  110010111  ←  mintermi []  (don't cares [407])
  111000010  ←  mintermi [450]
  111000011  ←  mintermi [451]
  111110101  ←  m

<IPython.core.display.Math object>

["ABCDEF'GH'I + A'B'C'E'FGH'I + A'B'C'E'FGHI' + B'C'DEF'GHI' + A'B'CDEF'G'I + A'B'CDEF'G'H + ABCD'E'F'G'H + A'BCDE'F'GH + ABC'D'EF'GI + ABC'D'EF'GH + AB'C'DEF'GH + ABCDEGHI' + A'B'C'D'FGH'"]